# 03 - Time Travel and Audit Verification

Notebook này chạy Part B - Task 3 ở chế độ **read-only**: không restore, không ghi vào Bronze/Silver/checkpoint. Version UPDATE/INSERT được tìm từ `DESCRIBE HISTORY` và `operationMetrics`, không đoán version.

Flow demo: history -> `versionAsOf` -> chọn MERGE -> so sánh before/after -> đọc `_delta_log/*.json` -> xác nhận latest version không đổi.

## Evidence nghiệm thu ngày 2026-09-30

| Case | Before | After | Evidence |
|---|---:|---:|---|
| UPDATE | 43 | 44 | `trip_id 93184dd0…`: fare `26.8 -> 30.8`, tip `5.86 -> 7.86` |
| INSERT | 44 | 45 | `trip_id c10b59a0…` không có ở v44; rows `45,849,822 -> 45,849,823` |
| Schema | 45 | 46 | `metaData` thêm `surcharge_fee`; bản ghi demo có `surcharge_fee = 1.5` |
| Protocol | — | v0 | `minReaderVersion = 1`, `minWriterVersion = 2` |

Latest version trước/sau audit đều là 46; audit chạy khoảng 40 giây. Số liệu trên là run đã lưu tại `docs/evidence/`; cell bên dưới tự discover lại từ bảng đang mở.

Lưu ý khi trình bày:
- Update demo được append vào Bronze **hai lần** (Bronze v13 fare 28.8, v14 fare 30.8) và cùng rơi vào microbatch 44; dedup giữ bản mới nhất nên chênh lệch là `+4.0/+2.0`.
- Commit v46 vừa thêm cột vừa UPDATE cùng trip (fare `30.8 -> 32.8`), vì chế độ `schema` của script demo cũng tăng fare/tip.

## 1. Khởi tạo Spark và đường dẫn

In [1]:
from pathlib import Path
import os
import sys


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'audit' / 'time_travel.py').exists():
            return candidate
    raise FileNotFoundError('Could not find the project root.')


ROOT = find_project_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Máy demo có RAM hạn chế; audit chỉ đọc file của các commit được chọn.
os.environ.setdefault('SILVER_DRIVER_MEMORY', '2g')

from pyspark.sql import functions as F
from src.audit.time_travel import commit_action_preview, run_audit
from src.silver.silver_pipeline import make_spark

SILVER = str(ROOT / 'data' / 'silver' / 'taxi_trips')
EVIDENCE = ROOT / 'logs' / 'task3_audit_notebook.json'  # không ghi đè evidence gốc
spark = make_spark('local[2]')
spark.sparkContext.setLogLevel('ERROR')
print('Spark:', spark.version)
print('Silver path:', SILVER)

Spark: 3.4.1
Silver path: D:\DSEB\SEM 7\Big Data\delta-lakehouse-project\data\silver\taxi_trips


## 2. DESCRIBE HISTORY

Mỗi MERGE tạo một commit JSON mới trong `_delta_log/` và một table version mới. Chỉ đọc metadata, không quét dữ liệu.

In [2]:
history_df = spark.sql(f"DESCRIBE HISTORY delta.`{SILVER}`")
(history_df
 .select('version', 'timestamp', 'operation',
         F.col('operationMetrics')['numSourceRows'].alias('source_rows'),
         F.col('operationMetrics')['numTargetRowsInserted'].alias('inserted'),
         F.col('operationMetrics')['numTargetRowsUpdated'].alias('updated'),
         F.col('operationMetrics')['numTargetFilesAdded'].alias('files_added'),
         F.col('operationMetrics')['numTargetFilesRemoved'].alias('files_removed'))
 .orderBy(F.desc('version'))
 .show(6, truncate=False))

+-------+-----------------------+---------+-----------+--------+-------+-----------+-------------+
|version|timestamp              |operation|source_rows|inserted|updated|files_added|files_removed|
+-------+-----------------------+---------+-----------+--------+-------+-----------+-------------+
|46     |2026-09-30 03:30:59.337|MERGE    |1          |0       |1      |1          |1            |
|45     |2026-09-30 03:23:51.484|MERGE    |1          |1       |0      |1          |0            |
|44     |2026-09-30 03:08:36.998|MERGE    |1          |0       |1      |1          |1            |
|43     |2026-09-29 04:42:57.003|MERGE    |2992       |0       |2992   |16         |16           |
|42     |2026-09-29 04:42:05.156|MERGE    |1033252    |1033252 |0      |16         |0            |
|41     |2026-09-29 04:39:29.882|MERGE    |1028729    |1028729 |0      |16         |0            |
+-------+-----------------------+---------+-----------+--------+-------+-----------+-------------+
only showi

## 3. `versionAsOf` đúng cú pháp trong đề

`version 0` là commit đầu tiên của Silver, trước mọi MERGE. Ở project này streaming chia initial load thành nhiều microbatch, nên v0 chỉ chứa microbatch đầu; vì vậy audit bên dưới chọn đúng version ngay trước từng CDC MERGE thay vì dùng v0 làm baseline.

In [3]:
df_past = spark.read.option('versionAsOf', 0).format('delta').load(SILVER)
df_now = spark.read.format('delta').load(SILVER)
for label, frame in (('versionAsOf 0', df_past), ('current', df_now)):
    print(f"{label:13}: {frame.count():,} rows | {len(frame.columns)} columns | "
          f"surcharge_fee={'surcharge_fee' in frame.columns}")

versionAsOf 0: 1,276,635 rows | 31 columns | surcharge_fee=False
current      : 45,849,823 rows | 32 columns | surcharge_fee=True


## 4. Audit end-to-end (read-only)

`run_audit` tự chọn commit CDC một dòng từ `operationMetrics`, đếm rows ở các `versionAsOf` liên quan, rồi so sánh file bị `remove` (thuộc snapshot trước) với file được `add` (snapshot sau) thay vì join toàn bộ ~46 triệu dòng. Nếu không tìm thấy bằng chứng, kết quả là `NOT VERIFIED` chứ không tạo dữ liệu giả.

In [4]:
evidence = run_audit(
    spark=spark,
    silver_path=SILVER,
    history_limit=100,
    schema_column='surcharge_fee',
    output=EVIDENCE,
)


=== SILVER TABLE HISTORY ===
v46 | 2026-09-30 10:30:59.337000 | MERGE | parameters={"matchedPredicates": "[{\"predicate\":\"((ingested_at#95 > ingested_at#5405) OR ((ingested_at#95 = ingested_at#5405) AND ((ingest_batch_id#94 > ingest_batch_id#5404) OR ((ingest_batch_id#94 = ingest_batch_id#5404) AND (raw_record_hash#96 > raw_record_hash#5406)))))\",\"actionType\":\"update\"}]", "predicate": "[\"(trip_id#5400 = trip_id#90)\"]", "notMatchedBySourcePredicates": "[]", "notMatchedPredicates": "[{\"actionType\":\"insert\"}]"} | metrics={"numOutputRows": "63863", "numTargetBytesAdded": "10062640", "numTargetRowsInserted": "0", "numTargetFilesAdded": "1", "numTargetRowsMatchedDeleted": "0", "numTargetFilesRemoved": "1", "numTargetRowsMatchedUpdated": "1", "executionTimeMs": "32324", "numTargetRowsCopied": "63862", "rewriteTimeMs": "10577", "numTargetRowsUpdated": "1", "numTargetRowsDeleted": "0", "scanTimeMs": "21541", "numSourceRows": "1", "numTargetChangeFilesAdded": "0", "numTargetRowsNot

## 5. `_delta_log/*.json` thô của các commit được chọn

Mỗi dòng là một action: `commitInfo` (operation, predicate, metrics), `remove` (file cũ bị thay thế), `add` (file mới), `metaData` (schema mới khi có schema evolution). Snapshot cũ vẫn đọc được vì file bị `remove` chưa bị xóa vật lý cho tới khi VACUUM.

In [5]:
schema_commit = evidence['schema_introduction']
selected = evidence['selected_versions']
versions = {selected['update_after_version'], selected['insert_after_version'],
            schema_commit['version'] if schema_commit else None} - {None}
for version in sorted(versions):
    print(f'--- _delta_log/{version:020d}.json')
    for line in commit_action_preview(SILVER, version, max_chars=200):
        print(line)
    print()

--- _delta_log/00000000000000000044.json
{"commitInfo": {"timestamp": 1790737716924, "operation": "MERGE", "operationParameters": {"predicate": "[\"(trip_id#5621 = trip_id#89)\"]", "matchedPredicates": "[{\"predicate\":\"((ingested_at#94 > i ...
{"remove": {"path": "part-00011-47b3389c-61f9-4686-842b-5722de512d41-c000.snappy.parquet", "deletionTimestamp": 1790737716814, "dataChange": true, "extendedFileMetadata": true, "partitionValues": {},  ...
{"add": {"path": "part-00011-d80c602c-a6b6-411e-80fa-1def51de4485-c000.snappy.parquet", "partitionValues": {}, "size": 10062172, "modificationTime": 1790737716764, "dataChange": true, "stats": "{\"num ...

--- _delta_log/00000000000000000045.json
{"commitInfo": {"timestamp": 1790738631422, "operation": "MERGE", "operationParameters": {"predicate": "[\"(trip_id#5621 = trip_id#89)\"]", "matchedPredicates": "[{\"predicate\":\"((ingested_at#94 > i ...
{"add": {"path": "part-00004-3057ed3c-14f4-4751-a6bf-257e52e95470-c000.snappy.parquet", "partitio

## 6. Tóm tắt để trình bày

In [6]:
print('Selected versions :', evidence['selected_versions'])
print('Rows per version  :', evidence['snapshot_counts'])
print('Protocol          :', evidence['protocol']['log_action'], 'from version', evidence['protocol']['log_version'])
print('Schema value      :', evidence['schema_value_record'])
print('Latest before/after:', evidence['latest_before']['version'], '/', evidence['latest_after']['version'])
print('Checks            :', evidence['checks'])
print('Result            :', evidence['verification'], f"({evidence['runtime_seconds']} s)")
print('Evidence JSON     :', EVIDENCE)

Selected versions : {'update_before_version': 43, 'update_after_version': 44, 'insert_before_version': 44, 'insert_after_version': 45}
Rows per version  : {43: 45849822, 44: 45849822, 45: 45849823}
Protocol          : {'minReaderVersion': 1, 'minWriterVersion': 2} from version 0
Schema value      : {'trip_id': '93184dd0b5a98cc1efe037f60a0ab8f5f90c178bbb12809ac61b991e106c9c05', 'surcharge_fee': 1.5, 'fare_amount': 32.8, 'tip_amount': 8.86, 'ingest_batch_id': 'demo_20260930T032831965801', 'record_source': 'cdc_demo'}
Latest before/after: 46 / 46
Checks            : {'history_available': True, 'historical_version_readable': True, 'update_evidence': True, 'insert_evidence': True, 'schema_column_present': True, 'schema_value_present': True, 'delta_log_evidence': True, 'protocol_evidence': True, 'non_destructive': True}
Result            : PASS (28.5 s)
Evidence JSON     : D:\DSEB\SEM 7\Big Data\delta-lakehouse-project\logs\task3_audit_notebook.json


In [7]:
spark.stop()
print('Hoàn tất demo; Silver không bị thay đổi.')

Hoàn tất demo; Silver không bị thay đổi.
